# Setup

In [ ]:
pip install rpy2

In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R
install.packages("survival")
install.packages("survminer")
install.packages("lubridate")
install.packages("anytime")
install.packages("DataExplorer")
install.packages("patchwork")

In [ ]:
%%R
library(tidyverse)
library(survival)
library(survminer)
library(anytime)
library(lubridate)
library(DataExplorer)
library(patchwork)

In [ ]:
import os
import numpy as np
import pandas as pd
import pandas_profiling
import subprocess
import plotnine
import matplotlib.pyplot as plt
#import lifelines
#from lifelines import KaplanMeierFitter
from plotnine import *  # Provides a ggplot-like interface to matplotlib.
from IPython.display import display
from datetime import datetime

## Plot setup.
theme_set(theme_bw(base_size = 11)) # Default theme for plots.

def get_boxplot_fun_data(df):
  """Returns a data frame with a y position and a label, for use annotating ggplot boxplots.

  Args:
    d: A data frame.
  Returns:
    A data frame with column y as max and column label as length.
  """
  d = {'y': max(df), 'label': f'N = {len(df)}'}
  return(pd.DataFrame(data=d, index=[0]))

# NOTE: if you get any errors from this cell, restart your kernel and run it again.

In [ ]:
import os
bucket = os.getenv("WORKSPACE_BUCKET")
bucket

# Import Data for EAU Risk Stratification Ascertainment

In [ ]:
import pandas
import os

# This query represents dataset "EAU Risk stratification" for domain "condition" and was generated for All of Us Controlled Tier Dataset v8
dataset_63907718_condition_sql = """
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary,
        c_occurrence.condition_start_datetime,
        c_occurrence.condition_end_datetime,
        c_occurrence.condition_type_concept_id,
        c_type.concept_name as condition_type_concept_name,
        c_occurrence.stop_reason,
        c_occurrence.visit_occurrence_id,
        visit.concept_name as visit_occurrence_concept_name,
        c_occurrence.condition_source_value,
        c_occurrence.condition_source_concept_id,
        c_source_concept.concept_name as source_concept_name,
        c_source_concept.concept_code as source_concept_code,
        c_source_concept.vocabulary_id as source_vocabulary,
        c_occurrence.condition_status_source_value,
        c_occurrence.condition_status_concept_id,
        c_status.concept_name as condition_status_concept_name 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                    WHERE
                        concept_id IN (133729, 141758, 192440, 194079, 194993, 195072, 197036, 199065, 201606, 201675, 201820, 316866, 320128, 35623051, 3655085, 36717471, 36717576, 4000167, 4056621, 4059603, 4066995, 4074815, 4095850, 4109023, 4113451, 4116319, 4138253, 4159131, 4171974, 4194890, 4194891, 4210005, 4219152, 4220238, 4226266, 4235863, 4242843, 4243784, 42537741, 42538157, 42539502, 4318839, 4323988, 4340799, 4341633, 436940, 438688, 439002, 441267, 443865, 600861, 607997, 81893, 81902)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )) c_occurrence 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_standard_concept 
            ON c_occurrence.condition_concept_id = c_standard_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_type 
            ON c_occurrence.condition_type_concept_id = c_type.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
            ON c_occurrence.visit_occurrence_id = v.visit_occurrence_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` visit 
            ON v.visit_concept_id = visit.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_source_concept 
            ON c_occurrence.condition_source_concept_id = c_source_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` c_status 
            ON c_occurrence.condition_status_concept_id = c_status.concept_id"""

dataset_63907718_condition_df = pandas.read_gbq(
    dataset_63907718_condition_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_63907718_condition_df.head(5)

In [ ]:
conditions = dataset_63907718_condition_df[['person_id', 'standard_concept_name', 'condition_start_datetime']]

In [ ]:
import pandas
import os

# This query represents dataset "EAU Risk stratification" for domain "procedure" and was generated for All of Us Controlled Tier Dataset v8
dataset_63907718_procedure_sql = """
    SELECT
        procedure.person_id,
        procedure.procedure_concept_id,
        p_standard_concept.concept_name as standard_concept_name,
        p_standard_concept.concept_code as standard_concept_code,
        p_standard_concept.vocabulary_id as standard_vocabulary,
        procedure.procedure_datetime,
        procedure.procedure_type_concept_id,
        p_type.concept_name as procedure_type_concept_name,
        procedure.modifier_concept_id,
        p_modifier.concept_name as modifier_concept_name,
        procedure.quantity,
        procedure.visit_occurrence_id,
        p_visit.concept_name as visit_occurrence_concept_name,
        procedure.procedure_source_value,
        procedure.procedure_source_concept_id,
        p_source_concept.concept_name as source_concept_name,
        p_source_concept.concept_code as source_concept_code,
        p_source_concept.vocabulary_id as source_vocabulary,
        procedure.modifier_source_value 
    FROM
        ( SELECT
            * 
        FROM
            `""" + os.environ["WORKSPACE_CDR"] + """.procedure_occurrence` procedure 
        WHERE
            (
                procedure_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr       
                    WHERE
                        concept_id IN (4021108, 4029565, 4030148, 4042907, 4047947, 4064225, 4065430, 4078310, 4079713, 4096461, 4098879, 4119238, 4144721, 4165739, 4181781, 4200418, 4208341, 4233085, 4262797, 4289172, 4348166)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )) procedure 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_standard_concept 
            ON procedure.procedure_concept_id = p_standard_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_type 
            ON procedure.procedure_type_concept_id = p_type.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_modifier 
            ON procedure.modifier_concept_id = p_modifier.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.visit_occurrence` v 
            ON procedure.visit_occurrence_id = v.visit_occurrence_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_visit 
            ON v.visit_concept_id = p_visit.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_source_concept 
            ON procedure.procedure_source_concept_id = p_source_concept.concept_id"""

dataset_63907718_procedure_df = pandas.read_gbq(
    dataset_63907718_procedure_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_63907718_procedure_df.head(5)

In [ ]:
import pandas
import os

# This query represents dataset "EAU Risk stratification" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
dataset_63907718_person_sql = """
    SELECT
        person.person_id,
        person.gender_concept_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        person.race_concept_id,
        p_race_concept.concept_name as race,
        person.ethnicity_concept_id,
        p_ethnicity_concept.concept_name as ethnicity,
        person.sex_at_birth_concept_id,
        p_sex_at_birth_concept.concept_name as sex_at_birth,
        person.self_reported_category_concept_id,
        p_self_reported_category_concept.concept_name as self_reported_category 
    FROM
        `""" + os.environ["WORKSPACE_CDR"] + """.person` person 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id 
    LEFT JOIN
        `""" + os.environ["WORKSPACE_CDR"] + """.concept` p_self_reported_category_concept 
            ON person.self_reported_category_concept_id = p_self_reported_category_concept.concept_id"""

dataset_63907718_person_df = pandas.read_gbq(
    dataset_63907718_person_sql,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook")

dataset_63907718_person_df.head(5)

# Sort Data into Binary Risk Stratification

## Conditions

In [ ]:
%%R -i conditions
conditions_risk <- conditions %>%
  filter(
    !standard_concept_name %in% c(
      "Acute gonococcal cystitis",
      "Erb-Duchenne palsy as birth trauma"
    ),
    !str_detect(standard_concept_name, "Wedge")
  ) %>%
    mutate(
    date_time = condition_start_datetime,
    .keep = "unused") 


table(conditions_risk$standard_concept_name)

In [ ]:
%%R
str(conditions_risk)

## Procedures

In [ ]:
%%R -i dataset_63907718_procedure_df
procedures_for_risk <- dataset_63907718_procedure_df %>%
  select(person_id, standard_concept_name, procedure_datetime) %>%
  filter(
    !standard_concept_name %in% c(
      "Appendectomy",
      "Bilateral total nephrectomy",
      "Excision ampulla of Vater",
      "Excision of Meckel's diverticulum",
      "Laparoscopic appendectomy"
    ),
    !str_detect(standard_concept_name, "polypectomy")
  ) %>%
    mutate(
    date_time = procedure_datetime,
    .keep = "unused") 

table(procedures_for_risk$standard_concept_name)

In [ ]:
%%R
str(procedures_for_risk)

## Combine to create EAU Risk stratification

In [ ]:
%%R

eau_risk <- conditions_risk %>% 
    rbind(procedures_for_risk)

write.csv(eau_risk, 
          "eau_risk.csv",
          row.names = FALSE)

In [ ]:
!head eau_risk.csv

In [ ]:
!gsutil -m cp eau_risk.csv {bucket}/data/

In [ ]:
!gsutil -m ls {bucket}/data/

# Table of What Conditions

In [ ]:
import os
bucket = os.getenv("WORKSPACE_BUCKET")
bucket 

In [ ]:
!gsutil -m cp {bucket}/data/eau_risk.csv .
!gsutil -m cp {bucket}/data/eau_risk.csv .
